# Group 42

Main

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

from thymio import Thymio

import vision
import motion_controll
import filtering
import local_nav
import global_nav

In [ ]:
thymio = Thymio(pos_init=[0, 0], orient=0)
await thymio._connect_to_thymio_()

ConnectionRefusedError: [Errno 61] Connection refused

In [2]:
thymio.pos = [20,20]
thymio.orient = 0
goal = [22,25]
print(motion_controll.follow_path(thymio, goal))

angle_speed: 162.666476215536, left_speed: 37, right_speed: 362
False


In [3]:
thymio.stop()

In [7]:
await thymio.unlock()

Thymio unlocked


## Main Loop

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

from thymio import Thymio

import vision
import motion_controll
import filtering
import local_nav
import global_nav

end_pos, grid = vision.init_grid()
start_pos, start_orient = vision.get_pos()
path = global_nav.find_path()

thymio = Thymio(pos_init=start_pos, orient=start_orient)
await thymio._connect_to_thymio_()
while(True):
    thymio.update_ir()
    print(thymio.ir_sensors)
    is_object = local_nav.is_object()
    nav_mode_change = False
    
    if(not is_object and thymio.nav_mode=="LOCAL"):
        nav_mode_change = True
        thymio.nav_mode = "GLOBAL"
        path = global_nav.find_path()

    if(is_object):
        if(thymio.nav_mode=="GLOBAL"):
            nav_mode_change = True
            thymio.nav_mode = "LOCAL"
        local_nav.avoid_obstacle(thymio, grid=grid, path=path)
    
    if(thymio.nav_mode=="GLOBAL"):
        next_wp = path[0]
        wp_reached = motion_controll.follow_path(thymio, next_wp)
        if(wp_reached):
            path.pop(0)  # Supprime le waypoint atteint
            if len(path) == 0:  # Si plus de waypoints
                print("Destination atteinte!")
                break  # Sortir de la boucle
    
    pos_on_img, orient_on_img = vision.get_pos()
    filtering.filter_pos(thymio, pos_on_img, orient_on_img)



Thymio connected


In [10]:
a = [[1,2],[3,4]]
a[1][1]

4

In [ ]:
await thymio.update_ir()
print(thymio.ir_sensors[0:5])

In [3]:
thymio.set_motor_speeds([100, 100])
while True:
    await thymio.update_ir()
    if sum(thymio.ir_sensors) > 2000:
        thymio.stop()
        break
    

Next cell makes the Thymio robot move forward for 4 seconds and then stops each time the Forward button is pressed. Program stops when the Backward button is pressed.  
It is intended to collect data to compute the **velocity variance**.

In [13]:
await thymio.button_loop()

Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Forward pressed
Backward pressed


Using this program to measure (with a ruler) the distance travelled by the bot each time to see differencies despite constant time and compute the variance on **speed state**.

In [17]:
data_velocity_error=[142, 138, 138, 137, 139, 137, 138, 135, 142, 141] #distances in mm travelled at presumed same speed for a constant time
data_velocity_error=[x/thymio.DELTA_T for x in data_velocity_error] #distances divided by the constant time to get true velocities

mean_speed=np.mean(data_velocity_error) #mean speed in mm/s
print(mean_speed)
ratio_speed=100/mean_speed #from tests above, for a speed of 100 we get a mean speed of 34.675 mm/s

q_v = np.var(data_velocity_error) # variance on speed state
print(q_v)

34.675
0.300625


<img src="position_measurement.png" width="400">

From the camera we got a data set of XY position measurements from the same position to search for some differencies and compute **variances on XY states and measurements**.

In [ ]:
measurements_from_camera=np.array([[13.118, 13.303], [13.153, 13.317], [13.090, 13.307], [13.062, 13.255], [13.059, 13.281],
                                   [13.074, 13.321], [13.026, 13.273], [13.073, 13.295], [13.023, 13.270]])
x_measurements = [x[0] for x in measurements_from_camera]
y_measurements = [y[1] for y in measurements_from_camera]

var_x = np.var(x_measurements)
var_y = np.var(y_measurements)
print("x measurement variance is: ", var_x)
print("y measurement variance is: ", var_y)

q_x = var_x/2 #variance on x position state
r_x = var_x/2 #variance on x position measurement
q_y = var_y/2 #variance on y position state
r_y = var_y/2 #variance on y position measurement

x measurement variance is:  0.0015213333333333543
y measurement variance is:  0.00046133333333333155


From the camera we got a data set of left and right turn angles travelled by the bot each time to see differencies despite constant time and compute the variance on **angle state**.

In [ ]:
visionInstance = Vision()
visionInstance.getEnvironment()

cap = cv2.VideoCapture(0)
ret, frame = cap.read()
start_theta= visionInstance.getRobotPoseCameraFrame(thymio, frame)[2]

thymio.set_motor_speeds([0,0])
delta_t = 2
start_time=0
k=1
data_theta_pos=np.zeros(k)
data_theta_neg=np.zeros(k)

for i in range(k):

    thymio.set_motor_speeds([100,-100])
    start_time=time.time()
    while time.time() - start_time < delta_t:
        await asyncio.sleep(0.01)
    thymio.set_motor_speeds([0, 0])
    ret, frame = cap.read()
    on_going_theta=visionInstance.getRobotPoseCameraFrame(thymio, frame)[2]
    data_theta_pos[i]=on_going_theta-start_theta

    start_theta=on_going_theta
    thymio.set_motor_speeds([-100,100])
    start_time=time.time()
    while time.time() - start_time < delta_t:
        await asyncio.sleep(0.01)
    thymio.set_motor_speeds([0, 0])
    ret, frame = cap.read()
    on_going_theta=visionInstance.getRobotPoseCameraFrame(thymio, frame)[2]
    data_theta_neg[i]=on_going_theta-start_theta

print(data_theta_pos)
print(data_theta_neg)

    

Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 1
Expected 4 arena corners, but found 2
Expected 4 arena corners, but found 2
Expected 4 arena corners, but found 3
Expected 4 arena corners, but found 3
Expected 4 arena corners, but found 3
Expected 4 arena corners, but found 3
Expected 4 arena corners, but found 3
Expected 4 arena corners, but found 5
Expected 4 a

TypeError: unsupported operand type(s) for -: 'float' and 'NoneType'

Exception in thread Thread-4:
Traceback (most recent call last):
  File "/Users/eleonore/miniconda3/envs/mpc2025/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/Users/eleonore/miniconda3/envs/mpc2025/lib/python3.12/site-packages/tdmclient/tcp.py", line 73, in run
    packet = self.read_packet()
             ^^^^^^^^^^^^^^^^^^
  File "/Users/eleonore/miniconda3/envs/mpc2025/lib/python3.12/site-packages/tdmclient/tcp.py", line 66, in read_packet
    raise error
  File "/Users/eleonore/miniconda3/envs/mpc2025/lib/python3.12/site-packages/tdmclient/tcp.py", line 59, in read_packet
    packet_len = self.read_uint32()
                 ^^^^^^^^^^^^^^^^^^
  File "/Users/eleonore/miniconda3/envs/mpc2025/lib/python3.12/site-packages/tdmclient/tcp.py", line 46, in read_uint32
    b = self.io.read(4)
        ^^^^^^^^^^^^^^^
  File "/Users/eleonore/miniconda3/envs/mpc2025/lib/python3.12/site-packages/tdmclient/tcp.py", line 99, in read
    return self.socket.rec